In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown


load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9774.55, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2026-01-23 13:35:27", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-01-23 13:35:27", 10032.55]], "total_portfolio_value": 10032.55, "total_profit_loss": 32.54999999999927}'

In [5]:
account.report()

'{"name": "ed", "balance": 9774.55, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2026-01-23 13:35:27", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-01-23 13:35:27", 10032.55], ["2026-01-23 13:35:43", 9888.55]], "total_portfolio_value": 9888.55, "total_profit_loss": -111.45000000000073}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 75.15,
  'timestamp': '2026-01-23 13:35:27',
  'rationale': 'Because this bookstore website looks promising'}]

In [18]:
params = {"command": "uv", "args": ["run", "myaccounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [10]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of a given account name\n    Args:\n        name (str): The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name\n    Args:\n        name (str): The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock\n\n    Args:\n        name (str): The name of the account holder\n        symbol (str): The symbol of the stock\n        quantity (int): The quantity of shares to buy\n        rationale (str): The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {

In [11]:
instructions = "You are able to manage an account for a client, and answer questions abount the account"
request = "My name is Ed and my account is under the name Ed. What's my balance and holdings?"
model = "gpt-4.1-mini"

In [12]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", model=model, instructions=instructions, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ed, your current cash balance is $9,774.55. Your holdings include 3 shares of Amazon (AMZN). Is there anything specific you would like to do with your account?

In [13]:
from accounts_client import read_accounts_resource

In [14]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9774.55, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2026-01-23 13:35:27", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-01-23 13:35:27", 10032.55], ["2026-01-23 13:35:43", 9888.55], ["2026-01-23 18:26:01", 9825.55]], "total_portfolio_value": 9825.55, "total_profit_loss": -174.45000000000073}


In [15]:
from accounts import Account
Account.get("Ed").report()

'{"name": "ed", "balance": 9774.55, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2026-01-23 13:35:27", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-01-23 13:35:27", 10032.55], ["2026-01-23 13:35:43", 9888.55], ["2026-01-23 18:26:01", 9825.55], ["2026-01-23 18:41:30", 9924.55]], "total_portfolio_value": 9924.55, "total_profit_loss": -75.45000000000073}'

In [19]:
date_params = {"command": "uv", "args": ["run", "thetime_server.py"]}
async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [20]:
mcp_tools

[Tool(name='get_todaydate', description="Get today's date\n\n    ", inputSchema={'properties': {}, 'title': 'get_todaydateArguments', 'type': 'object'}, annotations=None)]

In [22]:
instructions = "You are great at fetching the latest date and time"
request = "What is today's date?"
model = "gpt-4.1-mini"

In [23]:
async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="date_teller", model=model, instructions=instructions, mcp_servers=[mcp_server])
    with trace("date_teller"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Today's date is January 23, 2026.